In [ ]:
"""
Prophet 산출물 조립 (prophet_features)

prophet/prophet/*.csv (segment_key별 개별 파일, 72/90개 존재)를 하나로
합쳐 XGBoost 피처 매트릭스에 바로 join할 수 있는 형태로 정리한다.
이미 계산된 값을 재사용하는 조립 작업이라(Prophet 팀이 이미 학습을
끝낸 고정된 예측값), 우리 쪽 train 컷오프와 무관하게 미리 만들어두는 게
합리적이다(speed_features/network_features/weather_features/incident_flag와
동일 카테고리).

정리 내용:
  1) segment_id + direction -> segment_key 컬럼 생성
  2) 스펙에 없는 컬럼 제거: yhat/yhat_lower/yhat_upper(현재 시점 예측,
     y_hat_t30 계열과 중복) / y_actual(우리 V_segment와 사실상 동일,
     중복). y_hat_t30/y_hat_lower_t30/y_hat_upper_t30만 남긴다.
  3) split(Train/Val/Test) 컬럼 검증: 72개 파일 전체가 동일한 날짜
     경계(Train ~2026-06-13 / Val 06-14~06-27 / Test 06-28~06-30)를
     쓰는지 확인 - 확인 결과 전부 동일하며, 이는 원래 파이프라인 설계
     문서의 분할 기준(최근 2주 Val, 최근 3일 Test)과 정확히 일치한다.
     따라서 이 split 컬럼을 XGBoost용으로 그대로 재사용해도 된다.
  4) 90개 segment_key 중 빠진 18개 목록을 별도로 기록해둔다(성능이 안
     나와서 보류된 것으로 확인됨 - 추후 Prophet 담당자가 채워줄 때까지
     이 18개는 y_hat_t30 계열이 결측인 채로 XGBoost에 들어가게 된다).

출력: output/features/prophet_features.parquet
  segment_key, timestamp, split, y_hat_t30, y_hat_lower_t30, y_hat_upper_t30
"""

import glob
import os
from pathlib import Path

import pandas as pd
import polars as pl

PROPHET_DIR = "./prophet/prophet"
SEGMENT_LINK_MAPPING_PATH = "./output/segment_link_mapping.csv"

OUTPUT_DIR = Path("./output/features")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# ==================================================================
# 1. 전체 파일 로드 + segment_key 부여 + 컬럼 정리
# ==================================================================

files = sorted(glob.glob(os.path.join(PROPHET_DIR, "*.csv")))
print(f"prophet 파일 수: {len(files)}")

KEEP_COLS = ["segment_key", "timestamp", "split", "y_hat_t30", "y_hat_lower_t30", "y_hat_upper_t30"]

frames = []
for f in files:
    df = pd.read_csv(f)
    df["segment_key"] = df["segment_id"] + "_" + df["direction"]
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    frames.append(df[KEEP_COLS])

prophet_df = pd.concat(frames, ignore_index=True)
print(f"합친 shape: {prophet_df.shape}")
print(f"segment_key 수: {prophet_df['segment_key'].nunique()}")
prophet_df.head()

In [ ]:
# ==================================================================
# 2. 누락 segment_key 확인
# ==================================================================

seg_map = pd.read_csv(SEGMENT_LINK_MAPPING_PATH, encoding="utf-8")
all_keys = set((seg_map["segment_id"] + "_" + seg_map["direction"]).unique())
present_keys = set(prophet_df["segment_key"].unique())
missing_keys = sorted(all_keys - present_keys)

print(f"전체 segment_key: {len(all_keys)}")
print(f"Prophet에 있는 segment_key: {len(present_keys)}")
print(f"누락된 segment_key({len(missing_keys)}개, 성능 미달로 보류됨):")
for k in missing_keys:
    print(f"  - {k}")

pd.Series(missing_keys, name="segment_key").to_csv(
    OUTPUT_DIR / "prophet_missing_segments.csv", index=False, encoding="utf-8-sig"
)

In [ ]:
# ==================================================================
# 3. split 경계 일관성 검증
# ==================================================================
# 72개 segment_key 전체가 동일한 날짜 경계를 쓰는지 확인. 하나라도 다르면
# 원래 파이프라인이 가정한 "XGBoost는 모든 구간에 동일한 달력 기간 적용"
# 원칙이 깨지므로 반드시 확인해야 함.

split_bounds = prophet_df.groupby("split")["timestamp"].agg(["min", "max"])
print("전체 데이터 기준 split 경계:")
print(split_bounds)

per_segment_bounds = prophet_df.groupby(["segment_key", "split"])["timestamp"].agg(["min", "max"]).reset_index()
n_unique_bounds = per_segment_bounds.groupby("split")[["min", "max"]].nunique()
print("\nsegment_key별 경계가 전부 동일한지 확인(각 split당 unique 값이 1이어야 함):")
print(n_unique_bounds)

all_consistent = (n_unique_bounds == 1).all().all()
print(f"\n전체 일관성: {'OK - 모든 segment_key가 동일한 split 경계 사용' if all_consistent else '불일치 발견, 확인 필요'}")
assert all_consistent, "segment_key별 split 경계가 다릅니다 - 원인 확인 필요"

In [ ]:
# ==================================================================
# 4. 저장
# ==================================================================

pl.from_pandas(prophet_df).write_parquet(OUTPUT_DIR / "prophet_features.parquet")

print(f"저장 완료: {(OUTPUT_DIR / 'prophet_features.parquet').resolve()}")
print(f"shape: {prophet_df.shape}")
print(f"결측 확인:")
print(prophet_df[["y_hat_t30", "y_hat_lower_t30", "y_hat_upper_t30"]].isnull().sum())

In [ ]:
"""
사용법 요약 (후속 XGBoost 피처 매트릭스 조립 시)

- key: segment_key + timestamp (10분 단위)
- split 컬럼을 그대로 XGBoost의 Train/Val/Test 분할로 재사용 가능
  (검증 완료 - 원래 설계 문서의 분할 기준과 일치).
- 18개 segment_key(prophet_missing_segments.csv)는 이 테이블에 아예
  없으므로, feature_df와 left join하면 해당 구간의 y_hat_t30 계열은
  전부 결측(null)이 된다. XGBoost는 결측을 자체 처리할 수 있으나(트리
  분기 시 결측 방향 학습), 이 18개 구간을 학습에서 아예 제외할지 결측
  허용하고 포함할지는 별도 결정이 필요하다.
"""
print("완료")